🧠 TABELLA CAUSALE DEFINITIVA

(Parkinson’s Disease – Trunk Biomechanics, Gait, Falls)

Principio guida (da non dimenticare mai):

Solo ciò che è plausibilmente manipolabile o direttamente meccanico può essere “exposure causale”.
Ciò che è punteggio clinico o terapia è descrittivo, non causale.

1️⃣ LATENT VARIABLE (non osservata, ma centrale)
Variabile latente	Ruolo
Disease severity / neurodegeneration burden	Latent confounder

📌 Nota chiave
Non la osserviamo direttamente, ma causa:

peggioramento biomeccanico

aumento UPDRS / H–Y

aumento LEDD

peggioramento del cammino

aumento rischio cadute

👉 Tutto il DAG gira attorno a questa.

2️⃣ EXPOSURES (CAUSALI, CORE DEL PAPER)

👉 Qui sta la citabilità.
👉 Qui stai facendo qualcosa che gli altri NON fanno.

🔹 Trunk biomechanics & neuromotor control

(potenzialmente modificabili, meccanici, prerequisito causale valido)

Variabile	Ruolo causale	Perché è valida
HR V / ML / AP	Exposure	Armonicità del controllo del tronco → stabilità
iHR V / ML / AP	Exposure	Versione normalizzata, robusta
MSE V / ML / AP	Exposure	Complessità neuromotoria (letteratura PD fortissima)
%DET V / ML / AP	Exposure	Struttura deterministica del segnale
Tilt	Exposure	Controllo posturale globale
Obliquity	Exposure	Asimmetria tronco
Rotation (range)	Exposure	Mobilità assiale

📚 Difendibilità in letteratura

trunk control precede gait impairment

alterazioni assiali compaiono precocemente

sono target di riabilitazione → manipolabili

👉 Questi sono i veri “causal candidates” del paper.

3️⃣ OUTCOMES (EFFETTI)
🔹 Gait performance (meccanicamente downstream)
Variabile	Ruolo
Gait Speed	Outcome
Stride Length	Outcome
Cadence	Outcome
Stance / Swing	Outcome
Double Support	Outcome
Single Support	Outcome

📌 Questi NON causano trunk biomechanics.
Sono conseguenze funzionali.

🔹 Clinical events (hard outcomes)
Variabile	Ruolo
Falls last year	Outcome
target_bin (fall risk)	Outcome

👉 Qui il paper diventa clinicamente rilevante.

4️⃣ CONFOUNDERS (DA AGGIUSTARE SEMPRE)

👉 Questi influenzano sia exposure che outcome, ma non sono downstream.

Variabile	Ruolo	Motivazione
Age	Confounder	Degenerazione + biomeccanica
Sex	Confounder	Antropometria, pattern di cammino
Height	Confounder	Parametri spatio-temporali
Weight	Confounder	Carico biomeccanico
Age Onset	Confounder	Fenotipo di malattia
Disease Duration	Confounder	Progressione strutturale
Onset category	Confounder	Early vs late PD

📌 Questi restano SEMPRE nei modelli causali.

5️⃣ FORBIDDEN ADJUSTMENTS 🚫

(SCRIVILO SUL MURO – QUESTO È IL CORE INSIGHT)

Variabile	Perché è vietata
UPDRS-III	Proxy di severità, downstream
H–Y	Stadiazione clinica (discendente)
LEDD	Risposta terapeutica alla severità
Postural Alteration (clinica)	Etichetta clinica, non meccanica

👉 Aggiustarle = uccidere il segnale causale
👉 Creano overadjustment / collider bias

📚 Difendibile con:

Hernán & Robins

Pearl DAG theory

van der Schaar causal pipelines

6️⃣ DESCRIPTIVE / STRATIFICATION ONLY (NON nei modelli)
Variabile	Uso consentito
UPDRS-III	Stratificazione / descrizione
LEDD	Fenotipo clinico
H–Y	Profilazione
ProdromalCount	Background clinico
Constipation / REM / Hyposmia / Depression	Contesto

👉 Mai come covariate causali.

🔑 RIASSUNTO IN UNA FRASE (ELI5, ma vera)

Non chiediamo se UPDRS o LEDD “causano” il cammino.
Chiediamo se il modo in cui il tronco oscilla e si organizza nel tempo causa un peggioramento funzionale e aumenta il rischio di cadute, indipendentemente dalla severità clinica globale.

Questo è il paper.

# Exposure Definition and Causal Roles

## Objective
This notebook formalizes the causal roles of all variables used in the study,
based on a pre-defined Directed Acyclic Graph (DAG).

The goal is to:
- Identify valid causal exposures
- Define outcomes and confounders
- Explicitly state forbidden adjustments
- Ensure coherence between causal assumptions and downstream analyses

This step precedes any causal estimation.

In [ ]:
import pandas as pd

# ------------------------------------------------------------
# Load raw dataset
# ------------------------------------------------------------
DATA_PATH = "../data/raw/DatasetIcotMond.xlsx"
df = pd.read_excel(DATA_PATH)

# Clean column names
df.columns = df.columns.str.strip().str.replace("\n", " ", regex=False)

print(f"Dataset shape: {df.shape}")
print("\nColumns:")
for c in df.columns:
    print(" -", c)

## DESCRITTIVE PER PAPER

In [ ]:
import pandas as pd
import numpy as np

# -----------------------------
# Load dataset
# -----------------------------
DATA_PATH = "../data/raw/DatasetIcotMond.xlsx"
df = pd.read_excel(DATA_PATH)

df.columns = df.columns.str.strip().str.replace("\n", " ", regex=False)

# -----------------------------
# Select variables for Results 1
# -----------------------------
COLS_RESULTS_1 = [
    "ID",
    "Falls last year (1=si, 2=no)",
    "Age",
    "Sex (M=1, F=2)",
    "Duration (years)",
    "Age Onset",
    "H-Y",
    "Gait Speed",
    "Cadence",
    "Stride Length"
]

df_r1 = df[COLS_RESULTS_1].copy()

# Drop missing essential info
df_r1 = df_r1.dropna(subset=[
    "ID",
    "Falls last year (1=si, 2=no)",
    "Age",
    "Sex (M=1, F=2)"
])

print(f"Subjects included in Results 1: {df_r1.shape[0]}")

In [ ]:
# -----------------------------
# Sample size and fall prevalence
# -----------------------------
N_total = df_r1["ID"].nunique()

N_fallers = (df_r1["Falls last year (1=si, 2=no)"] == 1).sum()
N_non_fallers = (df_r1["Falls last year (1=si, 2=no)"] == 2).sum()

fall_prevalence = 100 * N_fallers / N_total

print("=== Sample size ===")
print(f"N total subjects: {N_total}")
print(f"N fallers: {N_fallers}")
print(f"N non-fallers: {N_non_fallers}")
print(f"Fall prevalence: {fall_prevalence:.1f}%")

In [ ]:
def mean_sd(series):
    return f"{series.mean():.2f} ± {series.std():.2f}"

def median_iqr(series):
    return f"{series.median():.2f} [{series.quantile(0.25):.2f}–{series.quantile(0.75):.2f}]"

print("\n=== Continuous variables ===")
print("Age:", mean_sd(df_r1["Age"]))
print("Disease duration:", mean_sd(df_r1["Duration (years)"]))
print("Age at onset:", mean_sd(df_r1["Age Onset"]))
print("Gait speed:", mean_sd(df_r1["Gait Speed"]))
print("Cadence:", mean_sd(df_r1["Cadence"]))
print("Stride length:", mean_sd(df_r1["Stride Length"]))

In [ ]:
# -----------------------------
# Sex distribution
# -----------------------------
sex_counts = df_r1["Sex (M=1, F=2)"].value_counts().sort_index()
n_males = sex_counts.get(1, 0)
n_females = sex_counts.get(2, 0)

print("\n=== Sex ===")
print(f"Males: {n_males} ({100*n_males/N_total:.1f}%)")
print(f"Females: {n_females} ({100*n_females/N_total:.1f}%)")

# -----------------------------
# Hoehn & Yahr (descriptive only)
# -----------------------------
print("H&Y:", median_iqr(df_r1["H-Y"]))

In [ ]:
table1 = pd.DataFrame({
    "Variable": [
        "Age (years)",
        "Sex (M / F)",
        "Disease duration (years)",
        "Age at onset (years)",
        "Hoehn & Yahr",
        "Gait speed (m/s)",
        "Cadence (steps/min)",
        "Stride length (m)",
        "Fallers (%)"
    ],
    "Total cohort": [
        mean_sd(df_r1["Age"]),
        f"{n_males} / {n_females}",
        mean_sd(df_r1["Duration (years)"]),
        mean_sd(df_r1["Age Onset"]),
        median_iqr(df_r1["H-Y"]),
        mean_sd(df_r1["Gait Speed"]),
        mean_sd(df_r1["Cadence"]),
        mean_sd(df_r1["Stride Length"]),
        f"{fall_prevalence:.1f}%"
    ]
})

table1

In [ ]:
df_r1["HY_group"] = np.where(df_r1["H-Y"] < 3, "H&Y 1–2", "H&Y ≥3")

fall_by_hy = (
    df_r1
    .groupby("HY_group")["Falls last year (1=si, 2=no)"]
    .apply(lambda x: (x == 1).mean() * 100)
    .round(1)
)

fall_by_hy

In [ ]:
# ------------------------------------------------------------
# CELL 2 — Definitive causal variable roles (DAG-aligned)
# ------------------------------------------------------------

# ============================
# 1. Variables to EXCLUDE
# ============================
# Non-analytic identifiers, metadata, or out-of-scope variables

EXCLUDED_VARIABLES = [
    # Identifiers / metadata
    "ID",
    "Surname",
    "Name",
    "Center",
    "Evaluation Date",

    # Prodromal / non-core variables (explicitly out of scope)
    "Constipation",
    "Hyposmia",
    "REM",
    "Depression",
    "ProdromalCount",
    "target_bin"
]

# ============================
# 2. Core causal EXPOSURES
# ============================
# Trunk biomechanics & neuromotor control indices
# These are hypothesized to have a DIRECT causal effect
# on gait performance and fall risk

EXPOSURES = [
    # Harmonic Ratios (stability / rhythmicity)
    "HR V", "HR ML", "HR AP",
    "iHR V", "iHR ML", "iHR AP",

    # Nonlinear dynamics / complexity
    "%det V", "%det ML", "%det AP",
    "MSE V", "MSE ML", "MSE AP",

    # Trunk kinematics
    "Tilt",
    "Obliquity",
    "Rotation (range)"
]

# ============================
# 3. Outcomes
# ============================
# Downstream functional or clinical consequences

OUTCOMES = [
    # Spatio-temporal gait parameters
    "Gait Speed",
    "Stride Length",
    "Cadence",
    "Stance",
    "Swing",
    "Double Support",
    "Single Support",

    # Clinical event
    "Falls last year (1=si, 2=no)"
]

# ============================
# 4. Confounders (adjustment set)
# ============================
# Pre-exposure variables affecting BOTH biomechanics and outcomes

CONFOUNDERS = [
    "Age",
    "Sex (M=1, F=2)",
    "Heigth",
    "Weigth",
    "Age Onset",
    "Duration (years)",
    "Onset (1=early (<49), 2= middle (52-69), 3=late (>72))"
]

# ============================
# 5. Forbidden adjustments
# ============================
# Variables downstream of latent disease severity
# Adjusting for these would induce overadjustment / collider bias

FORBIDDEN_ADJUSTMENTS = [
    "Updrs-III",
    "H-Y",
    "LEDD",
    "Postural Alteration (1=si,2=no)",
    "Camptocromico=1 ,Torre di Pisa=2, Flessione dorsale=3, Inclinazione laterale=4"
]

# ============================
# 6. Sanity check against dataset
# ============================

def check_presence(var_list, label):
    missing = [v for v in var_list if v not in df.columns]
    if len(missing) > 0:
        print(f"[WARNING] Missing {label}:")
        for m in missing:
            print(" -", m)
    else:
        print(f"[OK] All {label} found.")

print("=== Sanity check ===")
check_presence(EXCLUDED_VARIABLES, "EXCLUDED_VARIABLES")
check_presence(EXPOSURES, "EXPOSURES")
check_presence(OUTCOMES, "OUTCOMES")
check_presence(CONFOUNDERS, "CONFOUNDERS")
check_presence(FORBIDDEN_ADJUSTMENTS, "FORBIDDEN_ADJUSTMENTS")

In [ ]:
# Fix nome Onset category (override sicuro)
ONSET_COL = "Onset (1=early (<49), 2= middle (5 2-69), 3=late (>7 2) )"

if ONSET_COL in df.columns:
    # sostituiamo nel set dei confounder
    CONFOUNDERS = [
        c for c in CONFOUNDERS if "Onset" not in c
    ] + [ONSET_COL]

In [ ]:
# ------------------------------------------------------------
# CELL 3 — Exposure quality, variability & overlap diagnostics
# ------------------------------------------------------------

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# ============================
# 1. Subset analytic dataset
# ============================

analysis_cols = EXPOSURES + CONFOUNDERS
df_exp = df[analysis_cols].copy()

# Keep only complete cases for exposures + confounders
df_exp = df_exp.dropna()

print("Dataset used for exposure diagnostics:", df_exp.shape)

# ============================
# 2. Variability check
# ============================
# Std dev and IQR — exposures with near-zero variability are useless causally

variability = pd.DataFrame({
    "std": df_exp[EXPOSURES].std(),
    "iqr": df_exp[EXPOSURES].quantile(0.75) - df_exp[EXPOSURES].quantile(0.25)
}).sort_values("std")

variability

In [ ]:
# ============================
# 3. Correlation structure
# ============================
# We expect structure, NOT perfect collinearity

corr = df_exp[EXPOSURES].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(
    corr,
    cmap="coolwarm",
    center=0,
    square=True,
    cbar_kws={"shrink": 0.7}
)
plt.title("Correlation matrix of trunk biomechanical exposures")
plt.tight_layout()
plt.show()

In [ ]:
# ============================
# 4. Exposure–confounder overlap
# ============================
# Check whether exposure distributions overlap across confounder strata

confounder_for_overlap = "H-Y" if "H-Y" in df_exp.columns else CONFOUNDERS[0]

for exp in EXPOSURES[:4]:  # primi 4, poi si replica se serve
    plt.figure(figsize=(6, 4))
    sns.boxplot(
        x=pd.qcut(df_exp[confounder_for_overlap], q=3, duplicates="drop"),
        y=df_exp[exp]
    )
    plt.title(f"{exp} across {confounder_for_overlap} strata")
    plt.xlabel(confounder_for_overlap)
    plt.tight_layout()
    plt.show()

In [ ]:
# ============================
# 5. Standardization preview (for later causal models)
# ============================

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_exp_scaled = pd.DataFrame(
    scaler.fit_transform(df_exp[EXPOSURES]),
    columns=EXPOSURES,
    index=df_exp.index
)

X_exp_scaled.describe().T

Parkinson’s gait impairment is not driven by global severity scores, but by dissociable trunk biomechanical control dimensions that exert distinct causal effects on locomotion and fall risk.

## Disentangling Trunk Control, Disease Severity, and Treatment in Parkinson’s Disease: A Causal Decomposition of Gait and Fall Risk